### Pre-Req: Disabling the IPython history

In [1]:
from IPython import get_ipython

ip = get_ipython()
if ip is not None and hasattr(ip, "history_manager"):
    ip.history_manager.enabled = False

# 1. Setup and Imports

In [2]:
!pip install requests
import requests
import json
from datetime import datetime
from IPython.display import display, Markdown
import json

# 2. Simple Memory Store

In [3]:
# Simple in-notebook memory

class FarmMemory:
    def __init__(self):
        self.data = {
            "crop": None,
            "location": None,
            "stage": None,
            "history": []  # list of past runs
        }

    def set_farm_details(self, crop, location, stage=None):
        self.data["crop"] = crop
        self.data["location"] = location
        self.data["stage"] = stage

    def update(self, entry: dict):
        """Append a new decision entry to history."""
        self.data["history"].append(entry)

    def get(self):
        return self.data

    def get_last(self):
        """Return the last decision, or None if no history."""
        if not self.data["history"]:
            return None
        return self.data["history"][-1]

    def pretty_history(self, n_last: int = 5):
        """Return a markdown-formatted summary of the last n decisions."""
        if not self.data["history"]:
            return "No history available yet."

        lines = []
        for entry in self.data["history"][-n_last:]:
            ts = entry["timestamp"]
            crop = entry["crop"]
            loc = entry["location"]
            ws = entry["weather_snapshot"]
            temp = ws["temp"]
            rain = ws["precip"]
            hum = ws["humidity"]
            actions = entry["actions_suggested"]

            # Top-level bullet for the run
            lines.append(f"- **[{ts}] {crop.title()} @ {loc.title()}**")
            # Weather line
            lines.append(f"  - Weather: {temp}°C, {rain} mm rain, {hum}% humidity")
            # Actions as numbered list
            lines.append(f"  - Actions:")
            for i, a in enumerate(actions, 1):
                lines.append(f"    {i}. {a}")

        return "\n".join(lines)

# 3. The Weather Tool

In [4]:
def fetch_weather(location):
    """
    Fetches the next 24h forecast for the given city.
    location: str (e.g., "Mumbai")
    """
    url = f"https://geocoding-api.open-meteo.com/v1/search?name={location}&count=1"
    loc_res = requests.get(url).json()

    if "results" not in loc_res:
        return {"error": "Location not found"}

    lat = loc_res["results"][0]["latitude"]
    lon = loc_res["results"][0]["longitude"]

    weather_url = (
        f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}"
        f"&hourly=temperature_2m,precipitation,relativehumidity_2m"
    )

    weather = requests.get(weather_url).json()
    return weather

# 4. Decision Logic

In [5]:
# Crop-specific agronomy thresholds
CROP_CONFIG = {
    "Wheat": {
        "heat_threshold": 32,
        "humidity_risk": 75,
        "water_need": "moderate"
    },
    "Tomato": {
        "heat_threshold": 30,
        "humidity_risk": 70,
        "water_need": "high"
    },
    "Rice": {
        "heat_threshold": 34,
        "humidity_risk": 80,
        "water_need": "very-high"
    },
    "Maize": {
        "heat_threshold": 33,
        "humidity_risk": 70,
        "water_need": "moderate"
    }
}

def decide_actions(weather, crop):
    actions = []
    explanations = []

    # 🔹 SAFETY CHECK: if hourly data is missing, fail gracefully
    if "hourly" not in weather:
        actions.append("Could not fetch detailed hourly weather data. Please try a nearby city name or check the spelling of the location.")
        explanations.append(
            "The weather API response did not contain the expected 'hourly' section, "
            "so normal rule-based evaluation could not be performed."
        )
        return {
            "actions": actions,
            "explanations": explanations
        }

    hourly = weather["hourly"]

    temp = hourly["temperature_2m"][0]
    precip = hourly["precipitation"][0]
    humidity = hourly["relativehumidity_2m"][0]

    # -------------------------
    # 1. Load crop config
    # -------------------------
    crop_cfg = CROP_CONFIG.get(crop, {
        "heat_threshold": 32,
        "humidity_risk": 75,
        "water_need": "moderate"
    })

    heat_th = crop_cfg["heat_threshold"]
    humid_th = crop_cfg["humidity_risk"]
    water_need = crop_cfg["water_need"]

    # -------------------------
    # 2. Rain / Irrigation Rule
    # -------------------------
    if precip < 1:
        if water_need == "high":
            actions.append("Irrigate today: No rain expected and this crop requires consistent moisture.")
            explanations.append(
                f"Precipitation is low ({precip} mm) and {crop} has high water demand, "
                "so irrigation is recommended."
            )
        elif water_need == "very-high":
            actions.append("Irrigate generously: Rain is unlikely and this crop has very high water demand.")
            explanations.append(
                f"Precipitation is low ({precip} mm) and {crop} is very water-hungry, "
                "so a stronger irrigation is recommended."
            )
        else:
            actions.append("Irrigate lightly: No rain expected and the crop has moderate water needs.")
            explanations.append(
                f"Precipitation is low ({precip} mm) and {crop} has moderate water demand, "
                "so a light irrigation is sufficient."
            )
    else:
        actions.append("Do NOT irrigate: Rain expected soon — conserve water.")
        explanations.append(
            f"Precipitation forecast is {precip} mm, which is enough that irrigation can be skipped "
            "to avoid over-watering and to conserve water."
        )

    # -------------------------
    # 3. Temperature Rule (Heat Stress)
    # -------------------------
    if temp > heat_th:
        actions.append(
            f"High temperature ({temp}°C): Provide shade or mulch to reduce heat stress for {crop}."
        )
        explanations.append(
            f"The current temperature {temp}°C is above the heat threshold ({heat_th}°C) for {crop}, "
            "so heat protection is recommended."
        )
    else:
        actions.append(
            f"Temperature is within safe range ({temp}°C) for {crop}."
        )
        explanations.append(
            f"The current temperature {temp}°C is below the heat threshold ({heat_th}°C) for {crop}, "
            "so no additional heat protection is required."
        )

    # -------------------------
    # 4. Humidity Rule (Pest / Fungal Risk)
    # -------------------------
    if humidity > humid_th:
        actions.append(
            f"Humidity is high ({humidity}%): Increased fungal/pest risk for {crop}. Improve airflow if possible."
        )
        explanations.append(
            f"Humidity is {humidity}%, which is above the risk level ({humid_th}%) for {crop}, "
            "so there is a higher chance of fungal or pest issues."
        )
    else:
        actions.append(
            f"Humidity is normal ({humidity}%): No major pest or fungal risk indicators."
        )
        explanations.append(
            f"Humidity is {humidity}%, which is below the risk level ({humid_th}%) for {crop}, "
            "so no special pest/fungal action is required today."
        )

    return {
        "actions": actions,
        "explanations": explanations
    }

# 5. The Agent Class

In [6]:
class FarmAgent:
    def __init__(self):
        self.memory = FarmMemory()

    def setup_farm(self, crop, location, stage=None):
        self.memory.set_farm_details(crop, location, stage)

    def run_agent(self):
        crop = self.memory.data["crop"]
        location = self.memory.data["location"]
        stage = self.memory.data["stage"]

        # 1. Tool use
        weather = fetch_weather(location)

        # 2. Planning (now returns actions + explanations)
        decision = decide_actions(weather, crop)
        actions = decision["actions"]
        explanations = decision["explanations"]

        # 3. Memory update (evaluation/logging)
        log_entry = {
            "timestamp": str(datetime.now()),
            "crop": crop,
            "location": location,
            "stage": stage,
            "weather_snapshot": {
                "temp": weather["hourly"]["temperature_2m"][0],
                "precip": weather["hourly"]["precipitation"][0],
                "humidity": weather["hourly"]["relativehumidity_2m"][0],
            },
            "actions_suggested": actions,
            "explanations": explanations
        }

        self.memory.update(log_entry)

        # Final structured output
        result = {
            "farm_plan": actions,
            "explanations": explanations,
            "current_stats": log_entry,
            "full_history": self.memory.get(),
        }

        return result

# 6. Interactive Chat Interface

### This multi-turn interface allows the agent to handle multiple crop/location queries in a single session while preserving and displaying decision history.

In [7]:
def chat_user(text):
    display(Markdown(f"<div style='color:#1a73e8; font-size:16px;'><b>👤 You:</b> {text}</div>"))

def chat_agent(text):
    display(Markdown(f"<div style='color:#188038; font-size:16px;'><b>🤖 AgriSense:</b> {text}</div>"))

# Create ONE agent instance so memory persists across multiple queries
agent = FarmAgent()

chat_agent("Hello! I’m AgriSense, your farm assistant. I can help you plan daily actions using live weather and your crop details.")

while True:
    # ---- Get crop/location/stage for this round ----
    chat_agent("Which crop would you like guidance for?")
    crop = input()
    chat_user(crop)

    chat_agent("Great! Which city or region is your farm located in?")
    location = input()
    chat_user(location)

    chat_agent("What is the current growth stage of this crop? (e.g., Seedling, Vegetative, Flowering, Harvest)")
    stage = input()
    chat_user(stage)

    # ---- Run the agent for this crop ----
    chat_agent("Thanks! Let me check the weather and prepare your personalized farm guidance...")

    agent.setup_farm(crop=crop, location=location, stage=stage)
    output = agent.run_agent()

    summary = output["current_stats"]
    actions = output["farm_plan"]
    explanations = output.get("explanations", [])

    # ---- Show today’s plan ----
    chat_agent("Here is your farm plan for today:")

    display(Markdown(f"""
### 🌾 **Farm Summary**
- **Crop:** {summary['crop']}
- **Location:** {summary['location']}
- **Stage:** {summary.get('stage', 'Not specified')}
- **Temperature:** {summary['weather_snapshot']['temp']}°C  
- **Rain:** {summary['weather_snapshot']['precip']} mm  
- **Humidity:** {summary['weather_snapshot']['humidity']}%

---

### 📌 **Recommended Actions**

AgriSense is recommending the following measures to be taken:
"""))

    if explanations:
        for i, e in enumerate(explanations, 1):
            display(Markdown(f"**{i}.** {e}"))

    # ---- Ask if user wants to see history ----
    chat_agent("\nWould you like to see a summary of your recent decisions and weather history? (yes/no)")
    see_hist = input("Show history? (yes/no): ").strip().lower()
    chat_user(see_hist)

    if see_hist.startswith("y"):
        hist_md = agent.memory.pretty_history(n_last=5)
        chat_agent("Here are your recent runs:")
        display(Markdown(hist_md))

    # ---- Ask if user wants to do another crop/location ----
    chat_agent("Would you like guidance for another crop or location? (yes/no)")
    again = input("Another query? (yes/no): ").strip().lower()
    chat_user(again)

    if not again.startswith("y"):
        chat_agent("Got it. Thanks for using AgriSense! 🌾")
        break


<div style='color:#188038; font-size:16px;'><b>🤖 AgriSense:</b> Hello! I’m AgriSense, your farm assistant. I can help you plan daily actions using live weather and your crop details.</div>

<div style='color:#188038; font-size:16px;'><b>🤖 AgriSense:</b> Which crop would you like guidance for?</div>

 Wheat


<div style='color:#1a73e8; font-size:16px;'><b>👤 You:</b> Wheat</div>

<div style='color:#188038; font-size:16px;'><b>🤖 AgriSense:</b> Great! Which city or region is your farm located in?</div>

...

- **[2025-12-03 18:08:18.957345] Wheat @ Ranchi**
  - Weather: 12.5°C, 0.0 mm rain, 89% humidity
  - Actions:
    1. Irrigate lightly: No rain expected and the crop has moderate water needs.
    2. Temperature is within safe range (12.5°C) for Wheat.
    3. Humidity is high (89%): Increased fungal/pest risk for Wheat. Improve airflow if possible.
- **[2025-12-03 18:10:12.441997] Tomato @ Pune**
  - Weather: 12.1°C, 0.0 mm rain, 99% humidity
  - Actions:
    1. Irrigate today: No rain expected and this crop requires consistent moisture.
    2. Temperature is within safe range (12.1°C) for Tomato.
    3. Humidity is high (99%): Increased fungal/pest risk for Tomato. Improve airflow if possible.
- **[2025-12-03 18:11:02.592200] Soyabean @ Ahmedabad**
  - Weather: 16.3°C, 0.0 mm rain, 80% humidity
  - Actions:
    1. Irrigate lightly: No rain expected and the crop has moderate water needs.
    2. Temperature is within safe range (16.3°C) for Soyabean.
    3. Humidity is high (80%): Increased fungal/pest risk for Soyabean. Improve airflow if possible.
- **[2025-12-03 18:12:07.331518] Maize @ Ranchi**
  - Weather: 12.5°C, 0.0 mm rain, 89% humidity
  - Actions:
    1. Irrigate lightly: No rain expected and the crop has moderate water needs.
    2. Temperature is within safe range (12.5°C) for Maize.
    3. Humidity is high (89%): Increased fungal/pest risk for Maize. Improve airflow if possible.

<div style='color:#188038; font-size:16px;'><b>🤖 AgriSense:</b> Would you like guidance for another crop or location? (yes/no)</div>

Another query? (yes/no):  no


<div style='color:#1a73e8; font-size:16px;'><b>👤 You:</b> no</div>

<div style='color:#188038; font-size:16px;'><b>🤖 AgriSense:</b> Got it. Thanks for using AgriSense! 🌾</div>